# Do ECG and skin conductance respond together?

**Before you begin:** Request access to the [HABS dataset](https://huggingface.co/datasets/HABSteam/multimodal-eeg-emotion). Download day-1 preprocessed emotion-video BDFs and matching `.record_map` files for at least 6 participants (ideally 12), and `labels_train.csv`. Place them in `sample/PREPROCESSED/P*/` and `sample/`, respectively, or change `DATA_ROOT` below. See the [notebook README](README.md) for environment setup. This notebook does not download data.

A small multimodal research example: **ECG + GSR features → CCA → held-out participant tests → interpretation**.

**Question:** Do ECG and skin conductance (GSR) share trial-specific information during emotional videos that generalizes to unseen participants?

1. Extract ECG and GSR features from 10 s video trials and inspect signal quality.
2. Learn CCA representations without arousal labels and inspect what they encode.
3. Test cross-modal correspondence on unseen participants against shuffled trials.
4. Use a simple arousal probe, interpret the results and suggest next steps.

**Trials and ratings:** BDF event markers define video intervals; `labels_train.csv` provides optional self-reported arousal for each trial. Ratings do not enter CCA training, and rating periods are excluded.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pyedflib
from IPython.display import display
from scipy.signal import butter, find_peaks, sosfiltfilt
from sklearn.cross_decomposition import CCA
from sklearn.decomposition import PCA
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import LeaveOneGroupOut

# Same working-directory convention as the other starting notebooks.
cwd = Path.cwd().resolve()
if (cwd / "starting_notebooks" / "bdf_loader.py").is_file():
    ROOT = cwd
elif cwd.name == "starting_notebooks" and (cwd / "bdf_loader.py").is_file():
    ROOT = cwd.parent
else:
    raise RuntimeError("Start the kernel in the repository root or starting_notebooks/.")

mne.set_log_level("ERROR")

## 1 · Select trials

Use up to 12 participants, keeping **one 10 s video interval per observation**. BDF markers define the intervals; `labels_train.csv` supplies optional trial-level arousal ratings. We exclude rating periods and do not fill missing ratings.

The preceding 2 s provide a GSR baseline, although that interval may contain activity from the previous rating. GSR responses are delayed, so this simple alignment is an assumption to revisit.

In [ ]:
DATA_ROOT = ROOT / "sample"                       # Or Path("/your/local/dataset")
BDF_ROOT = DATA_ROOT / "PREPROCESSED"
TRIAL_S = 10.0       # analysed window after video onset (s)
BASELINE_S = 2.0     # GSR reference window before video onset (s)
MAX_PARTICIPANTS = 12
MIN_TRIALS = 10      # participants with fewer usable trials are dropped

# Select only local files; we do not scan or load the full 385 GB dataset.
bdfs = sorted(BDF_ROOT.glob("P*/P*_day-1_task-emotion-video_multimodal.bdf"))[:MAX_PARTICIPANTS]
if len(bdfs) < 6:
    raise FileNotFoundError(f"Found {len(bdfs)} emotion-video BDFs under {BDF_ROOT}; download at least 6 (ideally 12).")
if not (DATA_ROOT / "labels_train.csv").is_file():
    raise FileNotFoundError(f"Download the dataset's labels_train.csv to {DATA_ROOT}.")

# Video start markers, their end markers and emotion categories (as in real-time_decoding.ipynb).
marker_key = pd.read_csv(ROOT / "starting_notebooks" / "markers.csv")
video_key = marker_key.loc[marker_key["Start Marker"] >= 10100].set_index("Start Marker")
end_marker = video_key["End Marker"].to_dict()

labels = pd.read_csv(DATA_ROOT / "labels_train.csv", usecols=[
    "cohort", "participant_id", "day", "start_marker", "rating_arousal"
])
labels["participant_id"] = labels["participant_id"].astype(str).str.strip()
labels["day"] = pd.to_numeric(labels["day"].astype(str).str.extract(r"(\d+)")[0], errors="coerce")
labels["start_marker"] = pd.to_numeric(labels["start_marker"], errors="coerce")
labels["rating_arousal"] = pd.to_numeric(labels["rating_arousal"], errors="coerce")
labels = labels.dropna(subset=["day", "start_marker"])
labels[["day", "start_marker"]] = labels[["day", "start_marker"]].astype(int)
keys = ["participant_id", "day", "start_marker"]
if labels.duplicated(keys).any():
    raise ValueError("Ambiguous trial markers in labels_train.csv; check participant/day/trial identifiers.")

# The alphabetical selection is not a random sample: report which cohorts it covers.
selected = [bdf.parent.name for bdf in bdfs]
cohorts = labels.loc[labels.participant_id.isin(selected)].groupby("participant_id").cohort.first()
print(f"Selected {len(bdfs)} recordings: {', '.join(selected)}")
print("Participants per cohort:", cohorts.value_counts().sort_index().to_dict())

## 2 · Extract features

For **ECG**, compute median heart rate, RR-interval variation and early-to-late heart-rate change. The 10 s RR estimate is *not* clinical HRV. For **GSR**, compute baseline-relative rise, peak and slope.

Read only ECG/GSR rather than loading all BDF channels. Reject trials with missing baseline, short duration, flat/clipped GSR or too few plausible ECG beats. The simple ECG detector needs visual checking.

In [ ]:
ECG_COLS = ["ecg_hr", "ecg_rr_sd", "ecg_hr_change"]
GSR_COLS = ["gsr_rise", "gsr_peak", "gsr_slope"]


def detect_beats(ecg, fs):
    """Approximate R-peak positions (samples): 5–20 Hz band-pass, rectify, smooth, find peaks."""
    sos = butter(2, (5, 20), btype="bandpass", fs=fs, output="sos")
    # Rectification also handles an inverted ECG trace.
    width = max(1, round(0.08 * fs))
    envelope = np.convolve(np.abs(sosfiltfilt(sos, ecg)), np.ones(width) / width, mode="same")
    peaks, _ = find_peaks(envelope, distance=round(0.4 * fs), prominence=np.std(envelope))
    return peaks


def ecg_features(ecg, fs):
    """Heart rate, RR variability and heart-rate change in one trial, or None if unreliable."""
    peaks = detect_beats(ecg, fs)
    rr = np.diff(peaks) / fs
    plausible = (rr >= 0.4) & (rr <= 1.5)          # 40–150 beats per minute
    hr, beat_s = 60 / rr[plausible], peaks[1:][plausible] / fs
    late = beat_s >= TRIAL_S / 2
    if plausible.sum() < 6 or late.all() or not late.any():
        return None                                 # no trustworthy estimate from this window
    return {"ecg_hr": np.median(hr),
            "ecg_rr_sd": 1000 * np.std(rr[plausible]),
            "ecg_hr_change": hr[late].mean() - hr[~late].mean()}


def gsr_features(gsr, fs):
    """GSR response relative to the BASELINE_S seconds before onset, in the recording's own units."""
    n_baseline = round(BASELINE_S * fs)
    response = gsr[n_baseline:] - gsr[:n_baseline].mean()
    seconds = np.arange(len(response)) / fs
    return {"gsr_rise": response[seconds >= 2].mean(),   # skin conductance responds after ~1–3 s
            "gsr_peak": response.max(),
            "gsr_slope": np.polyfit(seconds[::20], response[::20], 1)[0]}


def video_intervals(bdf):
    """Pair each video start marker with the immediately following end marker."""
    raw = mne.io.read_raw_bdf(bdf, preload=False, verbose="ERROR")
    events = pd.DataFrame({
        "time": raw.annotations.onset - raw.first_time,
        "code": pd.to_numeric(raw.annotations.description, errors="coerce"),
    }).dropna().reset_index(drop=True)
    raw.close()
    pairs = []
    for i in range(len(events) - 1):
        start, end = events.iloc[i], events.iloc[i + 1]
        marker = int(start.code)
        if marker in end_marker and end.code == end_marker[marker] and end.time > start.time:
            pairs.append((float(start.time), float(end.time), marker))
    return pairs

In [ ]:
rows, log, gsr_overview, ecg_example = [], [], {}, None
for bdf in bdfs:
    participant = bdf.parent.name
    intervals = video_intervals(bdf)
    with pyedflib.EdfReader(str(bdf)) as reader:
        names = reader.getSignalLabels()
        if "ECG" not in names or "GSR" not in names:
            raise ValueError(f"Missing ECG/GSR in {bdf.name}; inspect its record_map.")
        ecg_id, gsr_id = names.index("ECG"), names.index("GSR")
        ecg_fs, gsr_fs = reader.getSampleFrequency(ecg_id), reader.getSampleFrequency(gsr_id)
        gsr_all = reader.readSignal(gsr_id)          # one channel of the recording: small
        floor, ceiling = gsr_all.min(), gsr_all.max()
        gsr_overview[participant] = (gsr_all[::100], 100 / gsr_fs, intervals)
        for order, (start_s, end_s, marker) in enumerate(intervals):
            features = None
            if end_s - start_s < TRIAL_S - 0.1:
                status = "video shorter than 10 s"
            elif start_s < BASELINE_S:
                status = "no pre-video baseline"
            elif start_s + TRIAL_S > len(gsr_all) / gsr_fs:
                status = "window exceeds recording"
            else:
                gsr = gsr_all[round((start_s - BASELINE_S) * gsr_fs):round((start_s + TRIAL_S) * gsr_fs)]
                if np.mean(np.diff(gsr) == 0) > 0.5 or np.mean((gsr <= floor) | (gsr >= ceiling)) > 0.01:
                    status = "GSR flat or clipped"
                else:
                    ecg = reader.readSignal(ecg_id, round(start_s * ecg_fs), round(TRIAL_S * ecg_fs))
                    features = ecg_features(ecg, ecg_fs)
                    status = "kept" if features else "too few ECG beats"
            log.append({"participant_id": participant, "start_marker": marker, "status": status})
            if features:
                rows.append({"participant_id": participant, "day": 1, "start_marker": marker,
                             "trial_order": order,
                             **features, **gsr_features(gsr, gsr_fs)})
                if ecg_example is None:
                    ecg_example = (participant, marker, ecg, ecg_fs)

log = pd.DataFrame(log)
trials = pd.DataFrame(rows).merge(labels.drop(columns="cohort"), on=keys, how="left",
                                  validate="one_to_one", indicator="in_labels")
if not np.isfinite(trials[ECG_COLS + GSR_COLS].to_numpy()).all():
    raise ValueError("Non-finite feature values; inspect the trials table.")
print(f"Video intervals found: {len(log)}; usable trials: {len(trials)}")

## 3 · Check signal quality

Inspect exclusions and sample traces **before** interpreting a negative result. In this dataset the GSR channel sometimes has very large, negative values despite its conductance-unit label; its physical calibration must be verified. The ECG detector can also mistake artefacts for beats.

The models below use within-participant variation, but standardization does **not** repair invalid measurements.

In [ ]:
print("Trials by participant and quality decision:")
display(pd.crosstab(log.participant_id, log.status))

# Drop participants with too few usable trials.
too_few = trials.groupby("participant_id").size().loc[lambda n: n < MIN_TRIALS]
if len(too_few):
    print("Dropped participants:", too_few.to_dict())
trials = trials.loc[~trials.participant_id.isin(too_few.index)].reset_index(drop=True)
if trials.participant_id.nunique() < 6:
    raise RuntimeError("Too few usable participants; inspect the quality table.")
print(f"Kept {len(trials)} trials / {trials.participant_id.nunique()} participants; "
      f"{trials.rating_arousal.notna().sum()} have arousal ratings.")

# Illustrate the sensor-quality problem with a few recordings, not all of them.
bad = log.loc[log.status == "GSR flat or clipped", "participant_id"]
example_ids = list(dict.fromkeys(
    (bad.iloc[:1].tolist()) + [next(iter(gsr_overview)), list(gsr_overview)[-1]]
))
fig, axes = plt.subplots(len(example_ids) + 1, 1, figsize=(9, 2 * len(example_ids) + 2),
                         gridspec_kw={"height_ratios": [1] * len(example_ids) + [1.5]})
for ax, participant in zip(axes, example_ids):
    gsr, step_s, intervals = gsr_overview[participant]
    ax.plot(np.arange(len(gsr)) * step_s, gsr, lw=0.7)
    for start_s, end_s, _ in intervals:
        ax.axvspan(start_s, end_s, color="tab:orange", alpha=0.15)
    ax.set(ylabel=participant, xticks=[])
axes[0].set_title("GSR recordings: raw scales and potential saturation")
participant, marker, ecg, fs = ecg_example
peaks, seconds = detect_beats(ecg, fs), np.arange(len(ecg)) / fs
axes[-1].plot(seconds, ecg - np.median(ecg), lw=0.7)
axes[-1].plot(seconds[peaks], ecg[peaks] - np.median(ecg), "rx")
axes[-1].set(xlabel="Seconds after video onset", yticks=[],
             title=f"Example ECG beat detection: {participant}, marker {marker}")
plt.tight_layout()
plt.show()

## 4 · Fit shared representations

Standardize features **within each participant** to focus on trial-to-trial variation. This uses all *unlabelled* trials of a held-out participant as calibration: the CCA model transfers across participants, but this is **not** strictly calibration-free prediction.

For each leave-one-participant-out fold, fit CCA and PCA using the other participants. CCA learns correlated ECG/GSR projections; their scaled average defines a convenient joint embedding. PCA and original features are comparisons.

In [ ]:
z = trials.groupby("participant_id")[ECG_COLS + GSR_COLS].transform(lambda x: (x - x.mean()) / x.std())
x_ecg, x_gsr = z[ECG_COLS].to_numpy(), z[GSR_COLS].to_numpy()
x_both = np.column_stack([x_ecg, x_gsr])
if not np.isfinite(x_both).all():
    raise ValueError("A feature is constant within a participant; inspect the summary table.")

groups = trials.participant_id.to_numpy()
folds = list(LeaveOneGroupOut().split(x_both, groups=groups))
assert all(set(groups[train]).isdisjoint(groups[test]) for train, test in folds)


def fit_representations(train):
    """Fit PCA and CCA on the training participants only (no labels)."""
    pca = PCA(n_components=2).fit(x_both[train])
    cca = CCA(n_components=2, scale=False, max_iter=1000).fit(x_ecg[train], x_gsr[train])
    u_train, v_train = cca.transform(x_ecg[train], x_gsr[train])
    u_sd, v_sd = u_train.std(axis=0), v_train.std(axis=0)

    def transform(rows):
        """Every feature set compared later, for the given trial rows."""
        u, v = cca.transform(x_ecg[rows], x_gsr[rows])
        return {"ECG only": x_ecg[rows], "GSR only": x_gsr[rows], "Concatenation": x_both[rows],
                "PCA (2D)": pca.transform(x_both[rows]),
                "CCA shared (2D)": (u / u_sd + v / v_sd) / 2}
    return cca, transform


models = [fit_representations(train) for train, _ in folds]
print(f"{len(folds)} folds: each tests one participant on a model fitted to the other {len(folds) - 1}.")

## 5 · Interpret the learned representation

For **one fitted CCA model**, correlate the original features with their respective ECG or GSR score on the first canonical component. This illustrates what the representation captures *without using arousal labels*.

Feature correlations are descriptive, not causal contributions. The component sign is arbitrary, and this plot alone does not establish cross-modal correspondence.

In [ ]:
# Inspect one model on its held-out participant; do not mix axes across folds.
(_, test), (cca, _) = folds[0], models[0]
ecg_score, gsr_score = cca.transform(x_ecg[test], x_gsr[test])

feature_names = ["ECG · Heart rate", "ECG · RR variation", "ECG · HR change",
                 "GSR · Rise", "GSR · Peak", "GSR · Slope"]
feature_r = (
    [np.corrcoef(x_ecg[test, i], ecg_score[:, 0])[0, 1] for i in range(len(ECG_COLS))]
    + [np.corrcoef(x_gsr[test, i], gsr_score[:, 0])[0, 1] for i in range(len(GSR_COLS))]
)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.barh(feature_names, feature_r)
ax.axvline(0, color="gray", lw=0.8)
ax.set(xlim=(-1, 1), xlabel="Correlation with each modality's first CCA score",
       title=f"CCA component 1 · held-out {groups[test][0]}")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6 · Test cross-modal correspondence

For every held-out participant, correlate the first ECG and GSR CCA projections on **matched trials**. Shuffle GSR trial order within that participant to build a control distribution.

Shuffling removes exact pairs, but does **not** rule out common session drift or carry-over effects. Compare participants individually and check whether one person dominates the group result.

In [ ]:
def correlation(x, y):
    return float(np.corrcoef(x, y)[0, 1])


N_SHUFFLES = 1000
rng = np.random.default_rng(42)
per_participant, null = [], []
for (train, test), (cca, _) in zip(folds, models):
    u, v = cca.transform(x_ecg[test], x_gsr[test])
    u, v = u[:, 0], v[:, 0]
    observed = correlation(u, v)
    shuffled = np.array([correlation(u, rng.permutation(v)) for _ in range(N_SHUFFLES)])
    null.append(shuffled)
    per_participant.append({"participant": groups[test][0], "trials": len(test), "matched r": observed,
                            "shuffled 2.5%": np.percentile(shuffled, 2.5),
                            "shuffled 97.5%": np.percentile(shuffled, 97.5)})
per_participant = pd.DataFrame(per_participant)

group_r, group_null = per_participant["matched r"].mean(), np.mean(null, axis=0)
group_p = (1 + np.sum(group_null >= group_r)) / (1 + N_SHUFFLES)
print(f"Mean matched r = {group_r:+.3f}; shuffled-pair p = {group_p:.3f} "
      "(exploratory; assumes exchangeable trials)")
print(f"Participants with positive r: {(per_participant['matched r'] > 0).sum()} / {len(per_participant)}")
print(f"Mean r without the strongest participant: "
      f"{per_participant['matched r'].drop(per_participant['matched r'].idxmax()).mean():+.3f}")

fig, ax = plt.subplots(figsize=(8, 3.4))
positions = np.arange(len(per_participant))
ax.vlines(positions, per_participant["shuffled 2.5%"], per_participant["shuffled 97.5%"],
          color="tab:blue", alpha=0.4, lw=6, label="Shuffled-pair range")
ax.scatter(positions, per_participant["matched r"], color="black", zorder=3, label="Matched trials")
ax.axhline(0, lw=0.8, color="gray")
ax.set_xticks(positions, per_participant.participant, rotation=45)
ax.set(ylabel="Matched ECG–GSR correlation", title="Held-out participants (CCA component 1)")
ax.legend(fontsize="small")
plt.tight_layout()
plt.show()

## 7 · Probe downstream utility

As an **exploratory downstream probe**, classify each trial as above/below its participant's median *reported* arousal. Compare ECG, GSR, concatenation, PCA and CCA with a fixed logistic regression and majority-class baseline.

The held-out participant's ratings are needed to define this relative target. It is **not** a deployable, calibration-free arousal predictor; the probe only checks whether the representation contains useful information.

In [ ]:
median_rating = trials.groupby("participant_id").rating_arousal.transform("median")
y = trials.rating_arousal.gt(median_rating).astype(float).where(trials.rating_arousal.notna())
labelled, target = y.notna().to_numpy(), y.to_numpy()

scores = {}
for (train, test), (_, transform) in zip(folds, models):
    train, test = train[labelled[train]], test[labelled[test]]
    if len(test) < 5 or len(set(target[test])) < 2:
        continue
    features_train, features_test = transform(train), transform(test)
    baseline = DummyClassifier(strategy="most_frequent").fit(x_both[train], target[train])
    scores.setdefault("Majority class", []).append(
        balanced_accuracy_score(target[test], baseline.predict(x_both[test])))
    for name in features_train:
        probe = LogisticRegression(class_weight="balanced", max_iter=1000)
        probe.fit(features_train[name], target[train])
        scores.setdefault(name, []).append(
            balanced_accuracy_score(target[test], probe.predict(features_test[name])))

print(f"{labelled.sum()} rated trials; {len(scores['Majority class'])} held-out participants scored")
display(pd.Series({name: f"{np.mean(values):.3f} ± {np.std(values):.3f}"
                   for name, values in scores.items()}).reindex([
    "Majority class", "ECG only", "GSR only", "Concatenation", "PCA (2D)", "CCA shared (2D)"
]).to_frame("Arousal · balanced accuracy"))

## Interpretation and next steps

**Findings.** What physiological features are associated with the learned component? Do ECG and GSR show stronger correspondence for matched trials than for shuffled pairs? Is this relationship consistent across unseen participants? Does the shared representation offer an advantage over individual modalities for predicting reported arousal?

In the example run, shared ECG–GSR structure was modest and varied across participants, while CCA did not improve arousal prediction over GSR alone. **Shared physiological information and predictive usefulness are different questions.** Mixed or negative results are valuable when interpreted in light of the original hypothesis.

**Research extensions.** Explore whether the conclusions change with different physiological time alignments, feature choices, or participant groups. Which follow-up experiment would most clearly test your interpretation?

**Resources:** [`DATASET.md`](../DATASET.md) · [scikit-learn CCA](https://scikit-learn.org/stable/modules/generated/sklearn.cross_decomposition.CCA.html).
